# Ejercicio tienda — ETL y EDA
Base de una tienda con dos tablas: **Ventas** y **Compras**.
Completamos cada una a 10 registros, limpiamos los errores (ETL) y hacemos el análisis (EDA).

## Paso 0 · Cargar los datos y completar a 10 registros
Leo las hojas Ventas y Compras y agrego 5 registros nuevos a cada una.
En los nuevos dejé algunos errores a propósito para el ETL.

In [ ]:
import pandas as pd

# En Colab, descomenta para subir el archivo:
# from google.colab import files
# subido = files.upload()

ventas = pd.read_excel('Ejercicio_tienda.xlsx', sheet_name='Ventas')
compras = pd.read_excel('Ejercicio_tienda.xlsx', sheet_name='Compras')

In [ ]:
# registros nuevos de ventas (con errores)
ventas_nuevas = pd.DataFrame([
    [6, 'cinamonroll',            1, 2.0, 4, 4.0, 8,  4],
    [7, 'cheese cake cereza',     1, 2.0, 4, 4.0, 8,  4],
    [8, 'Tiramisu ',              1, 2.0, 4, 4.0, 8,  4],   # nombre con espacio/mayúscula
    [9, 'galletas mms chocolate', 2, 2.0, 4, None, None, None],  # venta sin precio
    [6, 'cinamonroll',            1, 2.0, 4, 4.0, 8,  4],   # fila duplicada
], columns=ventas.columns)

# registros nuevos de compras (con errores)
compras_nuevas = pd.DataFrame([
    [6, 'harina',      2, 3.0, 6.0],
    [7, 'huevos',      1, 8.0, 8.0],
    [8, 'azucar',      1, '5,5', 5.5],   # precio con coma (texto)
    [9, 'mantequilla', 1, 7.0, 7.0],
    [6, 'harina',      2, 3.0, 6.0],     # fila duplicada
], columns=compras.columns)

ventas = pd.concat([ventas, ventas_nuevas], ignore_index=True)
compras = pd.concat([compras, compras_nuevas], ignore_index=True)
ventas

## Paso 1 · Entendimiento inicial
❯ ¿Cuántas filas y columnas hay, y de qué tipo es cada una?

In [ ]:
ventas.shape, compras.shape

In [ ]:
ventas.dtypes

## Paso 2 · Calidad de datos
❯ ¿Qué errores hay que corregir antes de analizar?

In [ ]:
ventas.isnull().sum()

In [ ]:
ventas.duplicated().sum(), compras.duplicated().sum()

## Paso 3 · Limpieza (ETL)
Corrijo: nombres con espacios/mayúsculas, precio con coma, filas duplicadas y la venta sin precio.

In [ ]:
# ventas: nombres en minúscula y sin espacios, quitar duplicados y la venta sin precio
ventas['Producto'] = ventas['Producto'].str.strip().str.lower()
ventas = ventas.drop_duplicates()
ventas = ventas.dropna(subset=['Precio total'])

# compras: el precio con coma pasa a número, y quito duplicados
compras['Precio U'] = pd.to_numeric(compras['Precio U'].astype(str).str.replace(',', '.'), errors='coerce')
compras['Producto'] = compras['Producto'].str.strip().str.lower()
compras = compras.drop_duplicates()

ventas.shape, compras.shape

## Paso 4 · Análisis univariado
❯ ¿Cuánto se vendió y cuánta ganancia dejó la tienda?

In [ ]:
print('Unidades vendidas:', ventas['Cantidad'].sum())
print('Ingreso total:', ventas['Precio total'].sum())
print('Ganancia total:', ventas['Ganancia'].sum())
print('Gasto total en compras:', compras['Precio'].sum())

In [ ]:
ventas['Ganancia'].describe()

## Paso 5 · Análisis bivariado
❯ ¿Qué producto deja más ganancia y en qué se gasta más?

In [ ]:
ventas.groupby('Producto')['Ganancia'].sum().sort_values(ascending=False).plot(kind='bar', title='Ganancia por producto')

In [ ]:
compras.groupby('Producto')['Precio'].sum().sort_values(ascending=False).plot(kind='bar', title='Gasto por producto')

## Paso 6 · Conclusiones
❯ ¿Qué le dirías a la dueña de la tienda?

**Hallazgo 1:** El producto que más ganancia deja es el tiramisú (12 en total), seguido de la
galleta red (6). Las galletas de chocolate son las que menos aportan.

**Hallazgo 2:** La ganancia total de ventas (30) es menor que el gasto en compras (62), y varias
compras (maquillaje, gloss, lápiz de ojos) no son de la tienda.

**Calidad de datos:** En los registros nuevos había una venta duplicada, una venta sin precio, un
precio con coma (azúcar) y un nombre mal escrito (Tiramisu). Se corrigieron en el Paso 3.

**Hipótesis a validar:** Si la tienda se enfoca en el tiramisú y deja de comprar cosas que no son
del negocio, el balance debería mejorar.